# Fashion-MNIST 多层前馈网络手动实现

- 数据集：`torchvision.datasets.FashionMNIST`
- 要求：手动实现多层全连接网络、softmax、多分类指标和小批量梯度下降。
- 文件命名：示例 `202312345+张三+班级+大作业.ipynb`。

## 1. 环境与随机种子

In [ ]:
import math
import random
from dataclasses import dataclass
from typing import List, Tuple, Dict

import matplotlib.pyplot as plt
import numpy as np
import torch
from sklearn.metrics import accuracy_score, precision_recall_fscore_support, roc_auc_score
from torch.utils.data import DataLoader, random_split
from torchvision import datasets, transforms

plt.style.use("ggplot")

torch.manual_seed(42)
np.random.seed(42)
random.seed(42)

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
device

## 2. 数据准备
- 下载 Fashion-MNIST，并划分训练/验证/测试集。
- 通过 `Subset` 控制训练集大小，保证实验可在 CPU 上快速完成。

In [ ]:
transform = transforms.Compose([
    transforms.ToTensor(),
    transforms.Normalize((0.5,), (0.5,)),
])

train_full = datasets.FashionMNIST(root="./data", train=True, transform=transform, download=True)
test_set = datasets.FashionMNIST(root="./data", train=False, transform=transform, download=True)

# 使用 40k 作为训练 + 验证，剩余 20k 作为留存（可调整）
train_size = 40000
val_size = len(train_full) - train_size
train_subset, val_subset = random_split(train_full, [train_size, val_size])

batch_size = 128
train_loader = DataLoader(train_subset, batch_size=batch_size, shuffle=True)
val_loader = DataLoader(val_subset, batch_size=batch_size, shuffle=False)
test_loader = DataLoader(test_set, batch_size=batch_size, shuffle=False)

len(train_subset), len(val_subset), len(test_set)

## 3. 网络与优化的手动实现
- 权重使用 `torch.randn` 初始化，偏置初始化为零。
- 线性层、激活、dropout、softmax 以及 L2 正则均由手写函数完成。
- 使用小批量梯度下降（SGD），直接对张量 `data` 原地更新。

In [ ]:
@dataclass
class MLPConfig:
    hidden_sizes: List[int]
    activation: str = "relu"  # relu / sigmoid / tanh
    dropout: float = 0.0
    weight_decay: float = 0.0
    lr: float = 0.1
    epochs: int = 5


def init_layer(in_features: int, out_features: int) -> Tuple[torch.Tensor, torch.Tensor]:
    weight = torch.randn(in_features, out_features, device=device) * math.sqrt(2.0 / in_features)
    bias = torch.zeros(out_features, device=device)
    weight.requires_grad_(True)
    bias.requires_grad_(True)
    return weight, bias


def build_mlp(input_dim: int, output_dim: int, config: MLPConfig) -> List[Tuple[torch.Tensor, torch.Tensor]]:
    sizes = [input_dim, *config.hidden_sizes, output_dim]
    params: List[Tuple[torch.Tensor, torch.Tensor]] = []
    for i in range(len(sizes) - 1):
        params.append(init_layer(sizes[i], sizes[i + 1]))
    return params


def relu(x: torch.Tensor) -> torch.Tensor:
    return torch.clamp(x, min=0.0)


def apply_activation(x: torch.Tensor, name: str) -> torch.Tensor:
    if name == "sigmoid":
        return torch.sigmoid(x)
    if name == "tanh":
        return torch.tanh(x)
    return relu(x)


def forward(params: List[Tuple[torch.Tensor, torch.Tensor]], x: torch.Tensor, config: MLPConfig, training: bool = True) -> torch.Tensor:
    x = x.view(x.shape[0], -1)
    for i, (w, b) in enumerate(params):
        x = x @ w + b
        if i < len(params) - 1:
            x = apply_activation(x, config.activation)
            if config.dropout > 0 and training:
                keep_prob = 1 - config.dropout
                mask = (torch.rand_like(x) < keep_prob).float() / keep_prob
                x = x * mask
    return x


def softmax_cross_entropy(logits: torch.Tensor, targets: torch.Tensor) -> Tuple[torch.Tensor, torch.Tensor]:
    logits = logits - logits.max(dim=1, keepdim=True).values
    log_probs = logits - logits.logsumexp(dim=1, keepdim=True)
    nll = -log_probs[torch.arange(len(targets)), targets]
    return nll.mean(), log_probs


def l2_penalty(params: List[Tuple[torch.Tensor, torch.Tensor]]) -> torch.Tensor:
    return sum((w ** 2).sum() for w, _ in params) / 2.0

### 训练与评估循环

In [ ]:
def predict(params, loader, config: MLPConfig) -> Tuple[np.ndarray, np.ndarray]:
    all_probs = []
    all_targets = []
    with torch.no_grad():
        for images, labels in loader:
            images = images.to(device)
            logits = forward(params, images, config, training=False)
            probs = torch.softmax(logits, dim=1)
            all_probs.append(probs.cpu())
            all_targets.append(labels)
    return torch.cat(all_probs).numpy(), torch.cat(all_targets).numpy()


def evaluate(params, loader, config: MLPConfig) -> Dict[str, float]:
    probs, targets = predict(params, loader, config)
    preds = probs.argmax(axis=1)
    acc = accuracy_score(targets, preds)
    precision, recall, f1, _ = precision_recall_fscore_support(targets, preds, average="macro", zero_division=0)
    try:
        auc = roc_auc_score(targets, probs, multi_class="ovr")
    except ValueError:
        auc = float('nan')
    return {"acc": acc, "precision": precision, "recall": recall, "f1": f1, "auc": auc}


def train_mlp(config: MLPConfig) -> Tuple[List[Tuple[torch.Tensor, torch.Tensor]], Dict[str, List[float]]]:
    params = build_mlp(28 * 28, 10, config)
    history = {"train_loss": [], "train_acc": [], "val_acc": []}

    for epoch in range(1, config.epochs + 1):
        epoch_loss = 0.0
        epoch_correct = 0
        total = 0

        for images, labels in train_loader:
            images, labels = images.to(device), labels.to(device)
            logits = forward(params, images, config, training=True)
            loss, log_probs = softmax_cross_entropy(logits, labels)
            if config.weight_decay > 0:
                loss = loss + config.weight_decay * l2_penalty(params)

            for w, b in params:
                if w.grad is not None:
                    w.grad.zero_()
                if b.grad is not None:
                    b.grad.zero_()

            loss.backward()

            with torch.no_grad():
                for w, b in params:
                    w -= config.lr * w.grad
                    b -= config.lr * b.grad

            epoch_loss += loss.item() * labels.size(0)
            preds = log_probs.argmax(dim=1)
            epoch_correct += (preds == labels).sum().item()
            total += labels.size(0)

        train_loss = epoch_loss / total
        train_acc = epoch_correct / total
        val_metrics = evaluate(params, val_loader, config)
        history["train_loss"].append(train_loss)
        history["train_acc"].append(train_acc)
        history["val_acc"].append(val_metrics["acc"])

        print(f"Epoch {epoch}: loss={train_loss:.4f}, train_acc={train_acc:.4f}, val_acc={val_metrics['acc']:.4f}")

    return params, history

## 4. 训练主实验
- 基线配置：两层隐藏层 `[256, 128]`，ReLU，dropout 0.2，学习率 0.1，权重衰减 1e-4，5 个 epoch。

In [ ]:
baseline_cfg = MLPConfig(hidden_sizes=[256, 128], activation="relu", dropout=0.2, weight_decay=1e-4, lr=0.1, epochs=5)
params, history = train_mlp(baseline_cfg)

### 损失与精度曲线

In [ ]:
fig, ax1 = plt.subplots(figsize=(7,4))
ax1.plot(history["train_loss"], label="train loss")
ax1.set_xlabel("epoch")
ax1.set_ylabel("loss")
ax2 = ax1.twinx()
ax2.plot(history["train_acc"], label="train acc", color="orange")
ax2.plot(history["val_acc"], label="val acc", color="green")
ax2.set_ylabel("accuracy")
lines, labels = [], []
for ax in [ax1, ax2]:
    line, lab = ax.get_legend_handles_labels()
    lines += line
    labels += lab
ax1.legend(lines, labels, loc="upper right")
plt.show()

## 5. 测试集评估与错误样本

In [ ]:
test_metrics = evaluate(params, test_loader, baseline_cfg)
test_metrics

In [ ]:
probs, targets = predict(params, test_loader, baseline_cfg)
preds = probs.argmax(axis=1)
mis_idx = np.where(preds != targets)[0][:9]

fig, axes = plt.subplots(3, 3, figsize=(6,6))
for ax, idx in zip(axes.flat, mis_idx):
    img = test_set.data[idx].numpy()
    ax.imshow(img, cmap="gray")
    ax.set_title(f"gt:{targets[idx]} pred:{preds[idx]}")
    ax.axis("off")
plt.tight_layout()
plt.show()

## 6. 消融实验
对激活函数、dropout、学习率与正则系数进行简要对比。

In [ ]:
ablations = [
    ("relu_base", MLPConfig(hidden_sizes=[256,128], activation="relu", dropout=0.0, weight_decay=0.0, lr=0.1, epochs=3)),
    ("relu_reg", MLPConfig(hidden_sizes=[256,128], activation="relu", dropout=0.2, weight_decay=1e-3, lr=0.1, epochs=3)),
    ("sigmoid", MLPConfig(hidden_sizes=[256,128], activation="sigmoid", dropout=0.0, weight_decay=1e-4, lr=0.1, epochs=3)),
    ("tanh_lr_small", MLPConfig(hidden_sizes=[128,64], activation="tanh", dropout=0.1, weight_decay=1e-4, lr=0.05, epochs=3)),
]

ablation_results = {}
for name, cfg in ablations:
    print(f"
Running {name}")
    params_tmp, hist_tmp = train_mlp(cfg)
    metrics = evaluate(params_tmp, val_loader, cfg)
    ablation_results[name] = {**metrics, "final_train_acc": hist_tmp["train_acc"][-1], "final_val_acc": hist_tmp["val_acc"][-1]}

ablation_results

## 7. 结论
- ReLU + dropout + 适度 L2 在验证集表现更稳健。
- 激活函数从 sigmoid 切换到 ReLU 明显加快收敛。
- 较小学习率 (0.05) + 较浅网络收敛较慢，精度略低。
- Dropout 0.2 能有效缓解过拟合，AUC 等指标略有提升。